<a href="https://colab.research.google.com/github/Sahilrai365/my-website/blob/main/Question_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [8]:
import json
with open('train-v1.1.json', 'r') as f:
    squad_data = json.load(f)

In [9]:
records = []

for article in squad_data['data']:
    title = article.get('title', '')
    for paragraph in article['paragraphs']:
        context = paragraph['context']
        for qa in paragraph['qas']:
            question = qa['question']
            qid = qa['id']
            for ans in qa['answers']:
                records.append({
                    'id': qid,
                    'title': title,
                    'context': context,
                    'question': question,
                    'answer': ans['text'],
                    'answer_start': ans['answer_start']
                })

# Convert to DataFrame
df = pd.DataFrame(records)
print(df.head())


                         id                     title  \
0  5733be284776f41900661182  University_of_Notre_Dame   
1  5733be284776f4190066117f  University_of_Notre_Dame   
2  5733be284776f41900661180  University_of_Notre_Dame   
3  5733be284776f41900661181  University_of_Notre_Dame   
4  5733be284776f4190066117e  University_of_Notre_Dame   

                                             context  \
0  Architecturally, the school has a Catholic cha...   
1  Architecturally, the school has a Catholic cha...   
2  Architecturally, the school has a Catholic cha...   
3  Architecturally, the school has a Catholic cha...   
4  Architecturally, the school has a Catholic cha...   

                                            question  \
0  To whom did the Virgin Mary allegedly appear i...   
1  What is in front of the Notre Dame Main Building?   
2  The Basilica of the Sacred heart at Notre Dame...   
3                  What is the Grotto at Notre Dame?   
4  What sits on top of the Main Building

In [10]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # remove excessive whitespace
    text = text.strip()
    return text

df['context'] = df['context'].apply(clean_text)
df['question'] = df['question'].apply(clean_text)

In [11]:
df['input_text'] = 'generate question: ' + df['context']
df['target_text'] = df['question']

In [12]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
model = T5ForConditionalGeneration.from_pretrained('t5-small')
tokenizer = T5Tokenizer.from_pretrained('t5-small')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [13]:
input_encodings = tokenizer(
    df['input_text'].tolist(),
    padding="longest",
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

target_encodings = tokenizer(
    df['target_text'].tolist(),
    padding="longest",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

# For training later (optional)
labels = target_encodings.input_ids
labels[labels == tokenizer.pad_token_id] = -100

In [14]:
from torch.utils.data import Dataset
class QGDataset(Dataset):
  def __init__(self,input_encodings,target_encodings):
    self.input_encodings = input_encodings
    self.target_encodings = target_encodings

  def __len__(self):
    return len(self.input_encodings['input_ids'])

  def __getitem__(self,idx):
    return {
        'input_ids': self.input_encodings.input_ids[idx],
        'attention_mask': self.input_encodings.attention_mask[idx],
        'labels': self.target_encodings.input_ids[idx]
    }

In [15]:
from torch.utils.data import DataLoader
dataset = QGDataset(input_encodings,target_encodings)
dataloader = DataLoader(dataset,batch_size=8,shuffle=True)

In [16]:
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    warmup_steps=500,
    fp16=True,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=500,
    save_steps=2000,
    save_total_limit=2,
)
trainer = Trainer(
    model=model,                    # The model you want to train
    args=training_args,             # Training arguments
    train_dataset=dataset,          # The dataset you want to train on
    tokenizer=tokenizer,            # Tokenizer for encoding inputs
)

# Start training
trainer.train()

<ipython-input-16-a441438bde5d>:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sahilrajarai26 (sahilrajarai26-manipal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
500,2.938600
1000,2.370100
1500,2.292200
2000,2.270700


Step,Training Loss
500,2.938600
1000,2.370100
1500,2.292200
2000,2.270700
2500,2.270500
3000,2.208100
3500,2.214000
4000,2.188800
4500,2.179100
5000,2.174100


TrainOutput(global_step=32850, training_loss=2.0736714736729454, metrics={'train_runtime': 11256.2857, 'train_samples_per_second': 23.347, 'train_steps_per_second': 2.918, 'total_flos': 3.5567419401437184e+16, 'train_loss': 2.0736714736729454, 'epoch': 3.0})

In [19]:
import torch

def generate_question(context):
    model.eval()
    input_text = "generate question: " + context
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model.generate(input_ids, max_length=64, num_beams=4, early_stopping=True)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)




In [20]:
print(generate_question("The capital of France is Paris."))
print(generate_question("Water boils at 100 degrees Celsius."))
print(generate_question("The Great Wall of China is one of the Seven Wonders of the World."))


What is the capital of France?
What temperature does water boil at?
What is one of the Seven Wonders of the World?


In [21]:
pip install rouge-score nltk


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f240f54026c57fdf7913c2a9a81372287ceefdda99d2ed3522eb0a5e2a798b32
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [22]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

reference = "What is the capital of France?"
generated = generate_question("The capital of France is Paris.")

score = scorer.score(reference, generated)
print("Generated:", generated)
print("ROUGE-L:", score['rougeL'].fmeasure)

Generated: What is the capital of France?
ROUGE-L: 1.0


In [25]:
model.save_pretrained('./qg-t5-small-finetuned')
tokenizer.save_pretrained('./qg-t5-small-finetuned')


('./qg-t5-small-finetuned/tokenizer_config.json',
 './qg-t5-small-finetuned/special_tokens_map.json',
 './qg-t5-small-finetuned/spiece.model',
 './qg-t5-small-finetuned/added_tokens.json')